# Exp 3 — Two-Phase Training (Freeze → Unfreeze)

**Hypothesis:** Freezing backbone for Phase 1 stabilises the head before full
fine-tuning, preventing catastrophic forgetting and achieving better final QWK.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve()))  # so 'from utils import *' works
from utils import *
print("utils loaded ✓")

In [ ]:
EXP_DIR   = ROOT / 'outputs' / 'swinv2_small_exp3'
CKPT_DIR_E = EXP_DIR / 'checkpoints'
FIG_DIR_E  = EXP_DIR / 'figures'
for d in (CKPT_DIR_E, FIG_DIR_E): d.mkdir(parents=True, exist_ok=True)
CKPT_P1   = CKPT_DIR_E / 'exp3_phase1_best.pt'
CKPT_PATH = CKPT_DIR_E / 'exp3_best.pt'

PHASE1_EPOCHS  = 5
PHASE2_EPOCHS  = 20   # extended for better convergence
PHASE1_HEAD_LR = 5e-4
PHASE2_HEAD_LR = 5e-4
PHASE2_BACK_LR = 3e-5  # lower than default — more stable unfreezing
GRAD_CLIP      = 1.0

In [ ]:
train_ds    = DRDataset(train_df, TRAIN_DIR, transform=enhanced_train_transform)
val_ds      = DRDataset(val_df,   VAL_DIR,   transform=val_transform)
test_ds     = DRDataset(test_df,  TEST_LABELS_DIR, transform=val_transform)
t_loader    = make_loader(train_ds, sampler=wrs_sampler)
v_loader    = make_val_loader(val_ds)
test_loader = make_val_loader(test_ds)

In [ ]:
# ── Phase 1: freeze backbone, train head only ─────────────────────────────────
set_seed()
model     = build_swin_model(pretrained=True).to(device)
criterion = FocalLoss(alpha=class_weights_t, gamma=2.0)
scaler    = torch.amp.GradScaler('cuda', enabled=use_amp)

for name, param in model.named_parameters():
    param.requires_grad = ('custom_head' in name)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Phase 1: {trainable:,} trainable params (custom_head only)')

optimizer  = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=PHASE1_HEAD_LR, weight_decay=WEIGHT_DECAY)
scheduler1 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE1_EPOCHS)

history_p1, _ = train_loop(model, criterion, optimizer, scheduler1, scaler,
                             t_loader, v_loader, CKPT_P1, epochs=PHASE1_EPOCHS)
print('Phase 1 complete.')

In [ ]:
# ── Phase 2: unfreeze all, layer-wise LR, grad clipping ───────────────────────
for param in model.parameters():
    param.requires_grad = True
print(f'Phase 2: {sum(p.numel() for p in model.parameters()):,} trainable params (all)')

ckpt_p1 = torch.load(CKPT_P1, map_location=device, weights_only=False)
model.load_state_dict(ckpt_p1['model_state'])
print(f'Loaded Phase 1 best (epoch {ckpt_p1["epoch"]}, κ={ckpt_p1["val_kappa"]:.4f})')

bp = [p for n,p in model.named_parameters() if 'custom_head' not in n and p.requires_grad]
hp = [p for n,p in model.named_parameters() if 'custom_head'     in n and p.requires_grad]
optimizer  = torch.optim.AdamW(
    [{'params':bp,'lr':PHASE2_BACK_LR},{'params':hp,'lr':PHASE2_HEAD_LR}],
    weight_decay=WEIGHT_DECAY)
scheduler2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE2_EPOCHS)

history_p2, best_kappa = train_loop(model, criterion, optimizer, scheduler2, scaler,
                                     t_loader, v_loader, CKPT_PATH, epochs=PHASE2_EPOCHS)
print(f'Phase 2 complete. Best κ = {best_kappa:.4f}')

In [ ]:
y_true, y_pred = evaluate(model, criterion, optimizer, scaler,
                           test_loader, CKPT_PATH, 'Exp3')
plot_confusion(y_true, y_pred, 'Exp3 Two-Phase', FIG_DIR_E / 'exp3_confusion.png', cmap='Purples')

import json as _json
results = {'qwk': cohen_kappa_score(y_true, y_pred, weights='quadratic'),
           'macro_f1': f1_score(y_true, y_pred, average='macro'),
           'f1_per': f1_score(y_true, y_pred, average=None).tolist(),
           'history_p1': history_p1, 'history_p2': history_p2}
with open(EXP_DIR / 'exp3_results.json', 'w') as f:
    _json.dump(results, f, indent=2)
print(f"Exp3 QWK: {results['qwk']:.4f}")